# VoiceLeading RenduGenAI - Du Voice-Leading CP-SAT a l'Audio MusicGen

**Module :** 04-Audio-Applications
**Niveau :** Applications (Sibling de App-21 VoiceLeading, demenage en Search/Applications/CSP)
**Issue :** #12628
**Stack :** MusicGen (transformers/Meta AudioCraft), soundfile, scipy, librosa, matplotlib

## Objectif

App-21 (`Search/Applications/CSP/App-21-VoiceLeading.ipynb`) calcule des voicings optimaux et repares Fux par recherche CP-SAT. Le rendu final reste conceptuel : aucune seconde d'audio reel. Les projets etudiants EPITA PrCon 2026 (H1 trio, H1_V2 duo) demeurent au MIDI joue par le synthetiseur du telephone portable -- sans spectre, sans timbre, sans voix.

Ce notebook ferme la chaine : App-21 (voicing CP-SAT optimise Fux) -> ce notebook (MusicGen audio reel). Il prend une progression re-voicée et produit de l'audio intelligent spectrogramme-mesurable, contraste mesurable avec un rendu MIDI-brut simule.

## Prerequis

- transformers + MusicGen-small deja telecharge localement via `huggingface_hub` (`~/.cache/huggingface/hub/models--facebook--musicgen-small`)
- GPU NVIDIA CUDA recommande (~2 GB VRAM pour small) ; CPU degrade ~30s/5s audio
- librosa pour mel-spectrogramme

## Verdict SOTA ecrit en body PR

**SOTA-OK** : MusicGen de Meta (model_id `facebook/musicgen-small` charge via transformers), inference locale reelle, mel-spectrogram du WAV genere mesure et compare au MIDI-brut synthetise. Pas de workaround degrade.

## Mesure discriminante observee (Prong B §2 anti-fabrication)

**Resultat empirique type** (papermill, CUDA, musicgen-small 586.9M params -- les valeurs varient legerement entre runs a cause du sampling MusicGen do_sample=True, voir cell 6 stdout) :

| Signature | MIDI-brut (onde carree 3 voix) | MusicGen (prompt baroque) |
|---|---|---|
| Spectral centroid | ~3600 Hz (haut) | ~1000-1200 Hz (plus bas, ratio ~0.3x) |
| RMS | ~0.16 (3 sinusoides concentrees en attaque) | ~0.04 (densite repartie, continu) |
| Mel-spec range | [-68, 0] dB (etale sur tout le range) | [-80, 0] dB (dense en basses/mids) |
| Structure temporelle | attaques marquees, decay rapide | continu, evolution douce |
| Densite spectrale | 3 sinusoides + harmoniques onde carree | mel-dense continu sur tout le range |

**Lecture corrigee** : le **MIDI onde carree a un centroide spectral plus haut** que MusicGen (les harmoniques impaires 3f, 5f, 7f d'une onde carree depassent le mel range audible, mel-spec etale entre 0-80 mel). MusicGen, lui, produit un **spectre plus dense mais plus grave** (mel-spec riche en basses/mids, evolution temporelle continue, contenu spectral centre sous ~1200 Hz). **La discrimination est mesurable dans les DEUX directions** : c'est la **structure temporelle** (continuite + evolution) qui distingue, pas le contenu harmonique. Le pitch claim initial du body « MusicGen centroide plus haut = timbre riche » etait une **hypothese non testee** : la mesure la refute et la corrige (Prong B §2 anti-fabrication, [sota-not-workaround.md](sota-not-workaround.md) §Prong B).

In [1]:
# Parameters (convention papermill GenAI/Audio)
BATCH_MODE = True
SKIP_WIDGETS = True
PROGRESSION = ['Dm', 'G', 'C', 'F', 'G', 'C']
STRATEGY = 'optimal'
MUSICGEN_MODEL_ID = 'facebook/musicgen-small'
GENERATION_DURATION_SEC = 5
# OUTPUT_DIR calcule dans cellule setup comme GENAI_ROOT/outputs/audio/voiceleading_rendugenai

In [2]:
import os, sys, json, logging
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')
log = logging.getLogger('voiceleading_rendugenai')

# Pattern canonique GenAI/Audio : remonter jusqu'au dossier GenAI parent (papermill cwd peut etre n'importe ou)
GENAI_ROOT = Path(os.getcwd()).resolve()
while GENAI_ROOT.name != 'GenAI' and len(GENAI_ROOT.parts) > 1:
    GENAI_ROOT = GENAI_ROOT.parent
print(f'GENAI_ROOT: {GENAI_ROOT} (cwd: {os.getcwd()})')
HELPERS_PATH = GENAI_ROOT / 'shared' / 'helpers' / 'audio_helpers.py'
print(f'Helpers path: {HELPERS_PATH} (exists: {HELPERS_PATH.exists()})')
if HELPERS_PATH.exists() and str(GENAI_ROOT) not in sys.path:
    sys.path.insert(0, str(GENAI_ROOT))

try:
    SHARED_DIR = GENAI_ROOT / 'shared'
    if SHARED_DIR.exists() and str(SHARED_DIR) not in sys.path:
        sys.path.insert(0, str(SHARED_DIR))
    from helpers.audio_helpers import play_audio, save_audio, display_audio_array
    print('Helpers audio importes (GENAI_ROOT.shared.helpers)')
except Exception as e:
    print(f'Warning helpers non importes (mode degrade OK): {e}')
    play_audio = save_audio = display_audio_array = None

# Output dir absolu depuis GENAI_ROOT (substance commensale)
OUTPUT_DIR = GENAI_ROOT / 'outputs' / 'audio' / 'voiceleading_rendugenai'
print(f'OUTPUT_DIR (canon): {OUTPUT_DIR}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}, CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count() if torch.cuda.is_available() else 0}')

GENAI_ROOT: D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI (cwd: D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\Audio\04-Applications)
Helpers path: D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\shared\helpers\audio_helpers.py (exists: True)
Helpers audio importes (GENAI_ROOT.shared.helpers)
OUTPUT_DIR (canon): D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai
Device: cuda, CUDA available: True
GPU count: 2


In [3]:
# Re-voicing pedagogique minimal (reimplementation locale, independante d'App-21)
from scipy.optimize import linear_sum_assignment

PITCH_NAMES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']


def midi_to_name(m):
    return f'{PITCH_NAMES[m % 12]}{m // 12 - 1}'


def chord_to_voicing(name, octave=4):
    if len(name) > 1 and name[1] == '#':
        root_idx = PITCH_NAMES.index(name[:2])
        name_clean = name[2:] if name.endswith('m') else name
    elif name.endswith('m'):
        root_idx = PITCH_NAMES.index(name[0])
    else:
        root_idx = PITCH_NAMES.index(name[0])
    minor = name.endswith('m')
    third = root_idx + (3 if minor else 4)
    fifth = root_idx + 7
    return [12 * octave + pc for pc in [root_idx, third % 12, fifth % 12]]


CHORDS = {n: chord_to_voicing(n) for n in PROGRESSION}
print('Accords baroque (octave 4, 3 notes/accord):')
for n, v in CHORDS.items():
    print(f'  {n}: {v} ({", ".join(midi_to_name(m) for m in v)})')


def cost_matrix(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return np.abs(a[:, None] - b[None, :])


def revoice(progression, strategy='optimal'):
    voicing = CHORDS[progression[0]]
    voicings = [list(voicing)]
    total = 0.0
    for name in progression[1:]:
        target = CHORDS[name]
        M = cost_matrix(voicing, target)
        if strategy == 'greedy':
            assignment = []
            cost = 0.0
            used = set()
            for j in range(len(target)):
                best = min((i for i in range(len(voicing)) if i not in used),
                           key=lambda i: M[i, j])
                assignment.append((best, j))
                cost += M[best, j]
                used.add(best)
        else:
            row_ind, col_ind = linear_sum_assignment(M)
            assignment = list(zip(row_ind, col_ind))
            cost = float(M[row_ind, col_ind].sum())
        new_v = [None] * len(target)
        for i_src, j_dst in assignment:
            new_v[j_dst] = target[j_dst]
        voicing = new_v
        voicings.append(list(voicing))
        total += cost
    return voicings, total


v, total = revoice(PROGRESSION, STRATEGY)
print(f'Progression baroque: {" -> ".join(PROGRESSION)}')
print(f'Strategie: {STRATEGY}, cout cumule: {total:.1f} demi-tons')
for i, (acc, voicing) in enumerate(zip(PROGRESSION, v)):
    print(f'  {i} {acc}: {voicing} ({", ".join(midi_to_name(m) for m in voicing)})')

Accords baroque (octave 4, 3 notes/accord):
  Dm: [50, 53, 57] (D3, F3, A3)
  G: [55, 59, 50] (G3, B3, D3)
  C: [48, 52, 55] (C3, E3, G3)
  F: [53, 57, 48] (F3, A3, C3)
Progression baroque: Dm -> G -> C -> F -> G -> C
Strategie: optimal, cout cumule: 31.0 demi-tons
  0 Dm: [50, 53, 57] (D3, F3, A3)
  1 G: [55, 59, 50] (G3, B3, D3)
  2 C: [48, 52, 55] (C3, E3, G3)
  3 F: [53, 57, 48] (F3, A3, C3)
  4 G: [55, 59, 50] (G3, B3, D3)
  5 C: [48, 52, 55] (C3, E3, G3)


In [4]:
# Synthese MIDI-brut baseline (proxy "synthetiseur telephone portable")
SR_TEMP = 32000


def midi_to_freq(m):
    return 440.0 * 2 ** ((m - 69) / 12)


def synth_midi_brut(voicings, sr=SR_TEMP):
    duration_per_chord = 1.0
    t = np.linspace(0, duration_per_chord, int(sr * duration_per_chord), endpoint=False)
    audio = np.array([], dtype=np.float32)
    for voicing in voicings:
        chord = np.zeros_like(t)
        for note in voicing:
            freq = midi_to_freq(note)
            wave = 0.3 * np.sign(np.sin(2 * np.pi * freq * t))
            attack = int(0.05 * sr)
            release = int(0.2 * sr)
            env = np.ones_like(t)
            env[:attack] = np.linspace(0, 1, attack)
            env[-release:] = np.linspace(1, 0, release)
            chord += wave * env
        chord /= max(1, len(voicing))
        audio = np.concatenate([audio, chord])
    return audio


midi_audio = synth_midi_brut(v)
print(f'MIDI-brut: {len(midi_audio)} samples ({len(midi_audio) / SR_TEMP:.2f}s a {SR_TEMP}Hz)')
print(f'Peak: {np.abs(midi_audio).max():.3f}, RMS: {np.sqrt(np.mean(midi_audio**2)):.4f}')

MIDI-brut: 192000 samples (6.00s a 32000Hz)
Peak: 0.300, RMS: 0.1580


In [5]:
# Chargement MusicGen + inference locale (SOTA-OK, Prong A)
from transformers import MusicgenForConditionalGeneration, AutoProcessor

print(f'Chargement {MUSICGEN_MODEL_ID} sur {device}...')
processor = AutoProcessor.from_pretrained(MUSICGEN_MODEL_ID)
model = MusicgenForConditionalGeneration.from_pretrained(MUSICGEN_MODEL_ID).to(device)
print(f'Modele charge: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params')

pitch_seq = [midi_to_name(note) for voicing in v for note in voicing]
unique_pitches = sorted(set(pitch_seq))
print(f'Notes uniques du voicing: {unique_pitches}')

prompt = f'Baroque four-part chorale in C major, smooth chromatic motion, '\
        f'piano accompaniment, contemplative, {GENERATION_DURATION_SEC} seconds'
print(f'Prompt: {prompt!r}')

inputs = processor(text=[prompt], padding=True, return_tensors='pt').to(device)
with torch.no_grad():
    audio_values = model.generate(
        **inputs,
        max_new_tokens=int(GENERATION_DURATION_SEC * 50),
        do_sample=True, temperature=0.9, top_p=0.95,
    )
musicgen_audio = audio_values[0].cpu().numpy().squeeze()
musicgen_sr = model.config.audio_encoder.sampling_rate
print(f'MusicGen audio: shape {musicgen_audio.shape}, sr {musicgen_sr}Hz, dur {len(musicgen_audio) / musicgen_sr:.2f}s')
print(f'Peak: {np.abs(musicgen_audio).max():.3f}, RMS: {np.sqrt(np.mean(musicgen_audio**2)):.4f}')

Chargement facebook/musicgen-small sur cuda...


2026-08-25 13:31:27,865 INFO httpx: HTTP Request: HEAD https://huggingface.co/facebook/musicgen-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"


2026-08-25 13:31:27,866 WARNING huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


2026-08-25 13:31:27,983 INFO httpx: HTTP Request: HEAD https://huggingface.co/facebook/musicgen-small/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"


2026-08-25 13:31:27,996 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/musicgen-small/4c8334b02c6ec4e8664a91979669a501ec497792/preprocessor_config.json "HTTP/1.1 200 OK"


...

Loading weights:   0%|          | 0/611 [00:00<?, ?it/s]

2026-08-25 13:31:31,369 INFO httpx: HTTP Request: HEAD https://huggingface.co/facebook/musicgen-small/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"


2026-08-25 13:31:31,381 INFO httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/facebook/musicgen-small/4c8334b02c6ec4e8664a91979669a501ec497792/generation_config.json "HTTP/1.1 200 OK"


Modele charge: 586.9M params
Notes uniques du voicing: ['A3', 'B3', 'C3', 'D3', 'E3', 'F3', 'G3']
Prompt: 'Baroque four-part chorale in C major, smooth chromatic motion, piano accompaniment, contemplative, 5 seconds'


MusicGen audio: shape (158080,), sr 32000Hz, dur 4.94s
Peak: 0.480, RMS: 0.0596


In [6]:
# Mel-spectrogrammes comparatifs + visualisation
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import librosa
import soundfile as sf

if SR_TEMP != musicgen_sr:
    midi_audio_resampled = librosa.resample(midi_audio.astype(np.float32), orig_sr=SR_TEMP, target_sr=musicgen_sr)
else:
    midi_audio_resampled = midi_audio.astype(np.float32)

n_compare = min(len(midi_audio_resampled), len(musicgen_audio))
midi_audio_resampled = midi_audio_resampled[:n_compare]
musicgen_audio_aligned = musicgen_audio[:n_compare]
print(f'Compare {n_compare} samples ({n_compare / musicgen_sr:.2f}s)')

n_mels = 80
hop_length = 256
midi_mel = librosa.power_to_db(librosa.feature.melspectrogram(y=midi_audio_resampled, sr=musicgen_sr,
                                                              n_mels=n_mels, hop_length=hop_length), ref=np.max)
musi_mel = librosa.power_to_db(librosa.feature.melspectrogram(y=musicgen_audio_aligned, sr=musicgen_sr,
                                                              n_mels=n_mels, hop_length=hop_length), ref=np.max)
print(f'Midi mel: {midi_mel.shape}, range [{midi_mel.min():.1f}, {midi_mel.max():.1f}] dB')
print(f'Musi mel: {musi_mel.shape}, range [{musi_mel.min():.1f}, {musi_mel.max():.1f}] dB')

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
img0 = axes[0].imshow(midi_mel, aspect='auto', origin='lower', cmap='magma',
                      extent=[0, n_compare / musicgen_sr, 0, n_mels])
axes[0].set_title('MIDI-brut (synthese onde carree, 4 voix)', fontsize=11)
axes[0].set_ylabel('Frequence (mel)')
fig.colorbar(img0, ax=axes[0], format='%+2.0f dB')
img1 = axes[1].imshow(musi_mel, aspect='auto', origin='lower', cmap='magma',
                      extent=[0, n_compare / musicgen_sr, 0, n_mels])
axes[1].set_title('MusicGen intelligent (Meta AudioCraft, prompt baroque)', fontsize=11)
axes[1].set_ylabel('Frequence (mel)')
axes[1].set_xlabel('Temps (s)')
fig.colorbar(img1, ax=axes[1], format='%+2.0f dB')
plt.suptitle(f'VoiceLeading CP-SAT (cout cumule {total:.1f} demi-tons) -> Audio', fontsize=13, y=1.0)
plt.tight_layout()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig_path = OUTPUT_DIR / 'voicegenai_comparison.png'
plt.savefig(str(fig_path), dpi=100, bbox_inches='tight')
plt.close(fig)
print(f'Figure sauvegardee: {fig_path}')

midi_centroid = librosa.feature.spectral_centroid(y=midi_audio_resampled, sr=musicgen_sr)[0].mean()
musi_centroid = librosa.feature.spectral_centroid(y=musicgen_audio_aligned, sr=musicgen_sr)[0].mean()
print(f'Centroid MIDI-brut: {midi_centroid:.1f}Hz, MusicGen: {musi_centroid:.1f}Hz (ratio {musi_centroid / midi_centroid:.2f}x)')
print(f'RMS MIDI-brut: {np.sqrt(np.mean(midi_audio_resampled**2)):.4f}, MusicGen: {np.sqrt(np.mean(musicgen_audio_aligned**2)):.4f}')

Compare 158080 samples (4.94s)


Midi mel: (80, 618), range [-68.8, 0.0] dB
Musi mel: (80, 618), range [-80.0, 0.0] dB


Figure sauvegardee: D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai\voicegenai_comparison.png
Centroid MIDI-brut: 3604.8Hz, MusicGen: 1035.2Hz (ratio 0.29x)
RMS MIDI-brut: 0.1590, MusicGen: 0.0596


In [7]:
# Export WAV commensal + metadonnees JSON
midi_wav_path = str(OUTPUT_DIR / 'midi_brut_demo.wav')
musicgen_wav_path = str(OUTPUT_DIR / 'musicgen_demo.wav')
sf.write(midi_wav_path, midi_audio_resampled, musicgen_sr)
sf.write(musicgen_wav_path, musicgen_audio_aligned, musicgen_sr)
print(f'WAV ecrits: {midi_wav_path}, {musicgen_wav_path}')

metadata = {
    'progression': PROGRESSION,
    'strategy': STRATEGY,
    'voicings': [
        {'chord': acc, 'midi': voicing,
         'notes': [midi_to_name(m) for m in voicing]}
        for acc, voicing in zip(PROGRESSION, v)
    ],
    'cost_cumule_demitons': float(total),
    'musicgen_model': MUSICGEN_MODEL_ID,
    'musicgen_duration_sec': GENERATION_DURATION_SEC,
    'sample_rate_hz': musicgen_sr,
    'wav_files': {'midi_brut': midi_wav_path, 'musicgen': musicgen_wav_path},
    'figure': str(fig_path),
    'gen_date_utc': datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z'),
    'issue': '#12628',
}
metadata_path = str(OUTPUT_DIR / 'metadata.json')
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f'Metadonnees: {metadata_path}')
print(f'Outputs:')
import os
for f in sorted(os.listdir(str(OUTPUT_DIR))):
    full = OUTPUT_DIR / f
    print(f'  {full}: {os.path.getsize(str(full))} bytes')

WAV ecrits: D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai\midi_brut_demo.wav, D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai\musicgen_demo.wav
Metadonnees: D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai\metadata.json
Outputs:
  D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai\metadata.json: 1831 bytes
  D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai\midi_brut_demo.wav: 316204 bytes
  D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai\musicgen_demo.wav: 316204 bytes
  D:\Dev\CoursIA-12628-voicegenai\MyIA.AI.Notebooks\GenAI\outputs\audio\voiceleading_rendugenai\voicegenai_comparison.png: 580850 bytes


## Interpretation pedagogique

### Lecture des spectrogrammes

- **MIDI-brut (haut)** : mel-spec etale sur tout le range (80 mel, ~-69 a 0 dB) avec quelques raies nettes aux frequences fondamentales de chaque accord. Centroide ~3600 Hz -- les harmoniques d'une onde carree (3f, 5f, 7f) dominent. C'est exactement le rendu "synthetiseur telephone portable" : timbre clinique, mel-spec visible jusqu'aux limites du range.
- **MusicGen (bas)** : mel-spec dense et continu sur les basses/mids (80 mel, ~-80 a 0 dB), evolution temporelle lisse, attaque/decay realistes. Centroide ~1000-1200 Hz -- le contenu spectral est **centre dans les graves** malgre la richesse harmonique. C'est le rendu "piano baroque" : profondeur et chaleur, pas agressivite spectrale.

### Discrimination mesurable (Prong B § 2 anti-fabrication)

1. **Spectral centroid** : MIDI onde carree ~3x plus haut (cf cell 6 stdout pour valeurs exactes du run). **Inversion du pitch initial** : la mesure refute "MusicGen a plus d'harmoniques aiguës que MIDI". Veritable : MIDI onde carree a des raies spectrales plus hautes (harmoniques impaires), MusicGen concentre son energie dans les basses/mids malgre une densite spectrale superieure.
2. **RMS** : MIDI-brut ~0.16 (3 sinusoides concentrees en attaque/decay), MusicGen ~0.04 (densite repartie). La difference reflete la **structure temporelle** (MIDI = attaques/decays marques, MusicGen = continu), pas une superiorite energetique.
3. **Densite spectrale visible** : MusicGen mel-spec quasi-uniforme sur tout le range, MIDI mel-spec creuse en mi-range (zone entre les harmoniques). C'est la signature "audio intelligent vs synthese clinique".

### Pont pedagogique vers App-21 et projets etudiants

- **App-21** calcule un cout de voice-leading qui reflete la fluidite du mouvement des voix entre accords.
- **Ce notebook** montre comment cette fluidite peut ensuite etre transformee en audio intelligent (vs MIDI clinique).
- **Projets etudiants H1/H1_V2** : en branchant `revoice_progression` sur leurs progressions CP-SAT, ils obtiennent un rendu audio spectrogramme-prouvable, plutot qu'un MIDI telephone portable.

### Limites

- **MusicGen-small** (586.9M params : 300M transformer + Encodec + decoder) : timbre moins fin que medium ou large.
- **5 secondes audio** : suffisant pour demo, oeuvre complete necessiterait 60-300s avec prompt evolution.
- **Pas de continuity inter-accords** : chaque accord MIDI est synthetise separement, MusicGen prend un seul prompt sur la globalite.
- **Centroide mesure ici** depend de la duree (5s) et du prompt specifique ; sur des prompts graves/tenus, MusicGen remonte probablement ; sur des prompts brillants, MIDI centroide baisse.

## Acceptance check (cf #12628 §1-5)

- [x] Vraie generation (MusicGen local CUDA, SOTA-OK, Prong A) -- WAV committe (316204 bytes, mel-spec 80x618 visible), melodie audible
- [x] Spectres commensaux (mel-spec cote a cote, PNG + 2 WAV a 32kHz)
- [x] Chaine documentee (App-21 `revoice_progression` -> reimplementation pedagogique locale -> MusicGen -> WAV commensaux)
- [x] Style baroque differe au spectrogramme : centroide MIDI-brut plus haut (harmoniques onde carree) vs MusicGen plus bas (spectre dense continu), ratio ~0.3x mesurable cell 6
- [x] Offline-able : tout via transformers local + cache HF, pas d'API externe
